# 20 RNN 与 LSTM

RNN 用来处理序列数据。它按时间一步步读入输入，并维护隐藏状态。LSTM 是 RNN 的改进版，更擅长保留长期信息。


## 0. 学习目标和阅读地图

RNN/LSTM 的重点是序列状态。你需要掌握：

1. 隐藏状态 `h_t` 如何携带过去信息。
2. 普通 RNN 为什么容易忘记长期依赖。
3. LSTM 的门控机制在解决什么问题。
4. 什么时候该考虑 Transformer 替代 RNN。


## 1. 数学逻辑

普通 RNN：

$$h_t = \tanh(W_xx_t + W_hh_{t-1}+b)$$

输出可以由最后一个隐藏状态得到：

$$\hat y = W_oh_T+b_o$$

LSTM 增加门控机制：

$$f_t=\sigma(W_f[x_t,h_{t-1}]+b_f)$$

$$i_t=\sigma(W_i[x_t,h_{t-1}]+b_i)$$

$$o_t=\sigma(W_o[x_t,h_{t-1}]+b_o)$$

门决定忘掉什么、写入什么、输出什么。


## 1.1 推导拆开看：为什么 RNN 会梯度消失

普通 RNN 反向传播时，梯度要穿过很多时间步。每一步都会乘上类似 `W_h` 和激活函数导数的项。

如果这些项的范数长期小于 1，梯度会指数级变小；如果长期大于 1，梯度可能爆炸。

LSTM 通过 cell state 和门控，让信息有更直接的路径跨越多个时间步，从而缓解长期依赖问题。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# 合成序列任务：长度为 12 的 0/1 序列，标签为前半段 1 的数量是否超过后半段
n, seq_len = 500, 12
X = np.random.randint(0, 2, size=(n, seq_len, 1)).astype('float32')
y = (X[:, :6, 0].sum(axis=1) > X[:, 6:, 0].sum(axis=1)).astype('int64')
print('X shape:', X.shape, 'y mean:', y.mean())


## 1.2 序列张量形状

本例的输入形状是：

$$batch \times seq\_len \times input\_size$$

也就是 `(样本数, 12, 1)`。每个样本是一段长度为 12 的 0/1 序列，每个时间步只有 1 个特征。

`nn.LSTM(..., batch_first=True)` 表示 batch 维度放在最前面。


In [ ]:
# 从零演示：RNN 隐藏状态如何逐步更新，不做完整训练
hidden_size = 4
Wx = np.random.normal(scale=0.5, size=(1, hidden_size))
Wh = np.random.normal(scale=0.5, size=(hidden_size, hidden_size))
b = np.zeros(hidden_size)

h = np.zeros(hidden_size)
example = X[0]
for t, xt in enumerate(example):
    h = np.tanh(xt @ Wx + h @ Wh + b)
    print(f't={t:2d} | x={int(xt[0])} | h={np.round(h, 3)}')


## 1.3 从零演示代码怎么读

NumPy 演示没有训练，只展示隐藏状态如何更新：

$$h_t=\tanh(x_tW_x+h_{t-1}W_h+b)$$

每读入一个 `x_t`，隐藏状态都会改变。理论上 `h_t` 包含了前面所有时间步的信息；实践中普通 RNN 很难长期保留。


In [ ]:
# PyTorch 实战：LSTM 做序列分类
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)
Xtr = torch.tensor(X_train, dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.long)
Xte = torch.tensor(X_test, dtype=torch.float32)

class LSTMClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=12, batch_first=True)
        self.head = nn.Linear(12, 2)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])

model = LSTMClassifier()
optimizer = torch.optim.Adam(model.parameters(), lr=0.02)
loss_fn = nn.CrossEntropyLoss()

for step in range(200):
    logits = model(Xtr)
    loss = loss_fn(logits, ytr)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    pred = model(Xte).argmax(dim=1).numpy()
print('LSTM accuracy:', round(accuracy_score(y_test, pred), 3))


In [ ]:
# 诊断：查看 LSTM 对几条测试序列的预测概率
with torch.no_grad():
    logits = model(Xte[:8])
    probs = torch.softmax(logits, dim=1).numpy()
for i in range(8):
    seq = ''.join(str(int(v)) for v in X_test[i, :, 0])
    print(f'seq={seq} | true={y_test[i]} | prob(class1)={probs[i, 1]:.3f}')


## 2.1 如何诊断序列模型

序列模型除了看 accuracy，还要看具体错误样本：模型是忽略前半段、后半段，还是对某类模式特别敏感？

如果序列很长，普通 LSTM 仍可能吃力。此时可以考虑 attention、Transformer、或者把序列特征做更好的摘要。


## 2. 常见误区

- RNN 按顺序处理，训练通常比 CNN/Transformer 难并且慢。
- 普通 RNN 容易梯度消失，LSTM/GRU 能缓解但不是万能。
- 序列最后一步不一定包含所有信息，任务不同可能需要 attention 或 pooling。

## 3. 小实验

- 改 `seq_len`，观察长序列难度。
- 把 LSTM 换成 `nn.GRU`。
- 增加 hidden size，看效果和速度。


## 5. 复习清单

- RNN 逐时间步更新隐藏状态。
- LSTM 用门控制遗忘、写入和输出。
- 长序列训练可能有梯度消失/爆炸。
- Transformer 更容易并行，也更擅长长依赖。
